In [ ]:
!pip install numpy matplotlib scikit-learn seaborn

# Lecture 05: Dimensionality Reduction with PCA

## Feature Extraction and Data Compression

## Big Idea
Principal Component Analysis (PCA) finds the directions of maximum variance in high-dimensional data and projects it onto a lower-dimensional subspace, preserving as much information as possible.

## Everyday Metaphor

> 💡 **PCA = finding the best angle to view a shadow**: Imagine a 3D object casting a shadow on a 2D wall. By changing the angle of the light, you can make the shadow show different aspects of the object. PCA finds the 'light angle' that shows the most informative 2D shadow of your high-dimensional data.
> 
> **High-dimensional data = complex object with many features**: Each feature is like a dimension describing the object (color, texture, size, weight, etc.).
> 
> **Principal components = optimal viewing angles**: The directions that capture the most variance in the data, like finding the best lighting to reveal the object's true shape.

## Why It Matters
- Reduces computational complexity by removing redundant features
- Helps visualize high-dimensional data in 2D or 3D
- Removes noise by focusing on directions with maximum signal
- Preprocessing step for many machine learning algorithms
- Essential for handling the 'curse of dimensionality' in modern datasets

## Formulas


### PCA Mathematical Foundation

Given centered data matrix \mathbf{X} (n samples × d features):

$$
\mathbf{X}_{centered} = \mathbf{X} - \frac{1}{n}\mathbf{1}\mathbf{1}^T\mathbf{X}
$$

$$
\text{Covariance matrix: } \quad \mathbf{\Sigma} = \frac{1}{n-1} \mathbf{X}_{centered}^T \mathbf{X}_{centered}
$$

$$
\mathbf{\Sigma} \mathbf{v}_i = \lambda_i \mathbf{v}_i
$$

*What each symbol means:*
- **\mathbf{X}**: Original data matrix
- **\mathbf{1}**: Vector of ones (for centering)
- **\mathbf{X}_{centered}**: Data with zero mean
- **\mathbf{\Sigma}**: Covariance matrix (d × d)
- **\mathbf{v}_i**: Eigenvector (principal component)
- **\lambda_i**: Eigenvalue (variance explained by PC i)


### PCA Projection

Project data onto first k principal components:

$$
\mathbf{Z} = \mathbf{X}_{centered} \mathbf{V}_k
$$

$$
\mathbf{X}_{approx} = \mathbf{Z} \mathbf{V}_k^T + \mathbf{\bar{x}}
$$

*What each symbol means:*
- **\mathbf{V}_k**: Matrix of first k eigenvectors (d × k)
- **\mathbf{Z}**: Projected data (n × k)
- **\mathbf{X}_{approx}**: Reconstructed data (n × d)
- **\mathbf{\bar{x}}**: Mean vector added back for reconstruction
- **k**: Number of principal components kept


### Variance Explained

$$
\text{Variance explained by PC } i: \quad \frac{\lambda_i}{\sum_{j=1}^d \lambda_j}
$$

$$
\text{Cumulative variance: } \quad \frac{\sum_{i=1}^k \lambda_i}{\sum_{j=1}^d \lambda_j}
$$

*What each symbol means:*
- **\lambda_i**: Eigenvalue of the i-th principal component
- **d**: Original number of features
- **k**: Number of components selected
- The eigenvalues represent variance along each principal direction


## Worked Example by Hand

> 🧮 Simple PCA: Reduce this 2D dataset to 1 dimension: [2,2], [2,4], [3,5], [4,4], [5,3], [5,5], [6,4]

### Step 1: Center the data

Calculate mean of each feature:
- Mean of x₁: (2+2+3+4+5+5+6)/7 = 27/7 ≈ 3.86
- Mean of x₂: (2+4+5+4+3+5+4)/7 = 27/7 ≈ 3.86

Centered data (subtract means):
[-1.86, -1.86], [-1.86, 0.14], [-0.86, 1.14], [0.14, 0.14], [1.14, -0.86], [1.14, 1.14], [2.14, 0.14]

### Step 2: Compute covariance matrix

$$
\mathbf{\Sigma} = \frac{1}{n-1} \sum_{i=1}^n (\mathbf{x}_i - \mathbf{\bar{x}})(\mathbf{x}_i - \mathbf{\bar{x}})^T
$$

After calculation:\n$$
\mathbf{\Sigma} = \begin{bmatrix} 2.24 & 0.76 \\ 0.76 & 1.81 \end{bmatrix}
$$

### Step 3: Find eigenvalues and eigenvectors

$$
det(\mathbf{\Sigma} - \lambda\mathbf{I}) = 0
$$

$$
(2.24 - \lambda)(1.81 - \lambda) - (0.76)^2 = 0
$$

$$
\lambda^2 - 4.05\lambda + 3.43 = 0
$$

Solving: $\lambda_1 ≈ 2.92, \lambda_2 ≈ 1.13$

Eigenvectors:
- For $\lambda_1 = 2.92$: $\mathbf{v}_1 ≈ [0.81, 0.59]$ (first principal component)
- For $\lambda_2 = 1.13$: $\mathbf{v}_2 ≈ [-0.59, 0.81]$ (second principal component)

### Step 4: Project onto first principal component

Projection: $z = \mathbf{x}_{centered} \cdot \mathbf{v}_1$

Example: For point [2,2] (centered [-1.86, -1.86]):
$$
z = [-1.86, -1.86] · [0.81, 0.59] = (-1.86×0.81) + (-1.86×0.59) ≈ -2.59
$$

### Step 5: Calculate variance explained

- PC1 explains: $2.92/(2.92+1.13) ≈ 2.92/4.05 ≈ 72.1%$
- PC2 explains: $1.13/(2.92+1.13) ≈ 1.13/4.05 ≈ 27.9%$

<em>Answer: The first principal component captures about 72% of the variance, so we can reduce from 2D to 1D while losing only about 28% of the information.</em>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits, load_breast_cancer
import pandas as pd


In [ ]:
# Generate sample datasets for PCA demonstration
# 1. Create correlated 2D data
np.random.seed(42)
n_samples = 200

# Create data with strong correlation
x1 = np.random.normal(0, 2, n_samples)
x2 = 0.7 * x1 + np.random.normal(0, 1, n_samples)  # Correlated with x1
X_2d = np.column_stack([x1, x2])

# 2. Load breast cancer dataset (higher dimensional)
cancer = load_breast_cancer()
X_cancer = cancer.data
y_cancer = cancer.target

print(f"2D data shape: {X_2d.shape}")
print(f"Breast cancer data shape: {X_cancer.shape}")
print(f"Number of features in cancer data: {X_cancer.shape[1]}")


In [ ]:
# PCA on 2D correlated data
# Standardize the data
scaler = StandardScaler()
X_2d_scaled = scaler.fit_transform(X_2d)

# Apply PCA
pca = PCA()
X_2d_pca = pca.fit_transform(X_2d_scaled)

# Plot results
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.scatter(X_2d_scaled[:, 0], X_2d_scaled[:, 1], alpha=0.6)
plt.title('Original 2D Data (Standardized)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.scatter(X_2d_pca[:, 0], X_2d_pca[:, 1], alpha=0.6)
plt.title('PCA Transformed Data')
plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
plt.bar(range(1, len(pca.explained_variance_ratio_) + 1), pca.explained_variance_ratio_)
plt.title('Explained Variance Ratio')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.xticks(range(1, len(pca.explained_variance_ratio_) + 1))

plt.tight_layout()
plt.show()

print(f"PCA components:\n{pca.components_}")
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Cumulative explained variance: {np.cumsum(pca.explained_variance_ratio_)}")


In [ ]:
# PCA for visualization: Breast cancer dataset
# Standardize the cancer data
scaler = StandardScaler()
X_cancer_scaled = scaler.fit_transform(X_cancer)

# Apply PCA to reduce to 2 components for visualization
pca_2d = PCA(n_components=2)
X_cancer_pca = pca_2d.fit_transform(X_cancer_scaled)

# Plot results colored by diagnosis
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_cancer_pca[:, 0], X_cancer_pca[:, 1], c=y_cancer, 
                    cmap='viridis', alpha=0.7)
plt.colorbar(scatter, label='Diagnosis (0=malignant, 1=benign)')
plt.title('PCA of Breast Cancer Dataset (2 components)')
plt.xlabel(f'First Principal Component ({pca_2d.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'Second Principal Component ({pca_2d.explained_variance_ratio_[1]:.1%} variance)')
plt.grid(True, alpha=0.3)
plt.show()

# Also show how many components needed for 95% variance
pca_full = PCA()
pca_full.fit(X_cancer_scaled)
cumsum = np.cumsum(pca_full.explained_variance_ratio_)
n_components_95 = np.argmax(cumsum >= 0.95) + 1
print(f"Components needed for 95% variance: {n_components_95}")
print(f"Explained variance with {n_components_95} components: {cumsum[n_components_95-1]:.3f}")


In [ ]:
# PCA reconstruction: How well can we reconstruct original data?
# Apply PCA with different numbers of components
pca_1 = PCA(n_components=1)
pca_2 = PCA(n_components=2)

# Fit and transform
X_pca_1 = pca_1.fit_transform(X_cancer_scaled)
X_pca_2 = pca_2.fit_transform(X_cancer_scaled)

# Inverse transform (reconstruct)
X_reconstructed_1 = pca_1.inverse_transform(X_pca_1)
X_reconstructed_2 = pca_2.inverse_transform(X_pca_2)

# Calculate reconstruction error (MSE)
mse_1 = np.mean((X_cancer_scaled - X_reconstructed_1) ** 2)
mse_2 = np.mean((X_cancer_scaled - X_reconstructed_2) ** 2)

print(f"Reconstruction error with 1 component: {mse_1:.4f}")
print(f"Reconstruction error with 2 components: {mse_2:.4f}")
print(f"Original data variance per feature: {np.var(X_cancer_scaled, axis=0).mean():.4f}")

# Visualize reconstruction for first few samples
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.scatter(X_cancer_scaled[:50, 0], X_cancer_scaled[:50, 1], alpha=0.7, label='Original')
plt.scatter(X_reconstructed_1[:50, 0], X_reconstructed_1[:50, 1], alpha=0.7, label='Reconstructed (1 PC)')
plt.xlabel('Feature 1 (scaled)')
plt.ylabel('Feature 2 (scaled)')
plt.title('Original vs Reconstructed (1 PC)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.scatter(X_cancer_scaled[:50, 0], X_cancer_scaled[:50, 1], alpha=0.7, label='Original')
plt.scatter(X_reconstructed_2[:50, 0], X_reconstructed_2[:50, 1], alpha=0.7, label='Reconstructed (2 PCs)')
plt.xlabel('Feature 1 (scaled)')
plt.ylabel('Feature 2 (scaled)')
plt.title('Original vs Reconstructed (2 PCs)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
components = [1, 2, 5, 10, 15, 20, 30]
errors = []
for n_comp in components:
    pca_temp = PCA(n_components=n_comp)
    X_pca_temp = pca_temp.fit_transform(X_cancer_scaled)
    X_reconstructed_temp = pca_temp.inverse_transform(X_pca_temp)
    mse_temp = np.mean((X_cancer_scaled - X_reconstructed_temp) ** 2)
    errors.append(mse_temp)

plt.plot(components, errors, 'o-', linewidth=2, markersize=8)
plt.xlabel('Number of Principal Components')
plt.ylabel('Reconstruction MSE')
plt.title('PCA Reconstruction Error vs Components')
plt.grid(True, alpha=0.3)
plt.show()


## Watch Out: Common Mistakes & Exam Traps

> ⚠️ **Common mistakes to avoid:**
> 1. **Forgetting to standardize data**: PCA is sensitive to scale; features with larger variances will dominate the first principal components.
> 2. **Misinterpreting principal components**: PCs are linear combinations of original features, not necessarily interpretable as individual features.
> 3. **Choosing too few components**: While dimensionality reduction is good, too few components can lose important information.
> 4. **Confusing PCA with factor analysis**: Both reduce dimensions but have different assumptions and goals (PCA maximizes variance, factor analysis models latent variables).
> 5. **Assuming PCs are always ordered by importance**: While PCs are ordered by explained variance, sometimes later PCs may be more interpretable or meaningful for specific applications.
> 6. **Overlooking that PCA is linear**: PCA cannot capture nonlinear relationships; consider Kernel PCA for nonlinear patterns.
> 7. **Using PCA on categorical data without proper encoding**: PCA assumes continuous, numeric data.
> 8. **Ignoring the reconstruction error**: Always check how well you can reconstruct original data from the reduced representation.


## Experiment Link

> 🔬 Run **Section 2: PCA Experiment** in this notebook to explore how PCA works on different datasets and understand the trade-offs in dimensionality reduction.

This section will include:
- PCA on various synthetic datasets (correlated, uncorrelated, high-dimensional)
- Visualization of data before and after PCA transformation
- Analysis of explained variance vs number of components
- Reconstruction error analysis
- Comparison with other dimensionality reduction techniques (t-SNE, LDA)
- Discussion of when to use PCA vs other methods


## Videos to Watch

> 📺 **Recommended video searches:**

1. **Principal Component Analysis (PCA) Clearly Explained**
   - Search: https://www.youtube.com/results?search_query=Principal+Component+Analysis+PCA+explained
   - Watch for: Intuitive explanation of what PCA does and why it works

2. **StatQuest: PCA main ideas in only 5 minutes!!!**
   - Search: https://www.youtube.com/results?search_query=StatQuest+PCA+main+ideas
   - Watch for: Quick, clear explanation of PCA concepts

3. **Linear Algebra for Machine Learning: PCA**
   - Search: https://www.youtube.com/results?search_query=linear+algebra+PCA+machine+learning
   - Watch for: Mathematical derivation and deeper understanding of PCA


## Practice Questions

> ❓ Try to solve these yourself first, then check the answers below!

### Question 1 (Concept): What is the main goal of PCA?

<details>
<summary>Click to reveal answer</summary>

The main goal of PCA is to reduce the dimensionality of data while preserving as much of the original variance as possible by projecting onto orthogonal directions of maximum variance.

</details>


### Question 2 (Concept): Why is it important to standardize data before applying PCA?

<details>
<summary>Click to reveal answer</summary>

PCA is sensitive to the scale of features because it seeks directions of maximum variance. Features with larger scales will have larger variances and dominate the first principal components, leading to biased results. Standardization ensures all features contribute equally.

</details>


### Question 3 (Calculation): Given the covariance matrix [[4, 2], [2, 3]], what are the eigenvalues?

<details>
<summary>Click to reveal answer</summary>

Solve det([[4-λ, 2], [2, 3-λ]]) = 0
(4-λ)(3-λ) - 4 = 0
λ² - 7λ + 8 = 0
Using quadratic formula: λ = (7 ± √(49-32))/2 = (7 ± √17)/2
λ₁ ≈ (7 + 4.12)/2 ≈ 5.56, λ₂ ≈ (7 - 4.12)/2 ≈ 1.44

</details>


### Question 4 (Concept): When would you choose to keep only the first principal component?

<details>
<summary>Click to reveal answer</summary>

Choose only the first PC when: it explains a large majority of the variance (e.g., >90%), you need extreme dimensionality reduction for visualization or computational efficiency, or the first PC captures the dominant pattern in your data while later PCs represent noise.

</details>


### Question 5 (Explain it simply): What does it mean when we say principal components are 'orthogonal'?

<details>
<summary>Click to reveal answer</summary>

Orthogonal means the principal components are perpendicular to each other in feature space. Mathematically, their dot product is zero. This ensures they capture independent aspects of the variance in the data.

</details>


## Cheat Sheet Draft

> 📝 This section is optimized for printing - it should fit approximately 1/7 of an A4 page when printed.

- Optimized for printing - fits approximately 1/7 of an A4 page
- Contains key formulas, definitions, and common mistakes
- Designed for quick review and reference

### Key Formulas

$$
\mathbf{X}_{centered} = \mathbf{X} - \frac{1}{n}\mathbf{1}\mathbf{1}^T\mathbf{X}
$$

*Data centering*

$$
\mathbf{\Sigma} = \frac{1}{n-1} \mathbf{X}_{centered}^T \mathbf{X}_{centered}
$$

*Covariance matrix*

$$
\mathbf{\Sigma} \mathbf{v}_i = \lambda_i \mathbf{v}_i
$$

*Eigenvalue decomposition*

$$
\mathbf{Z} = \mathbf{X}_{centered} \mathbf{V}_k
$$

*Projection onto k components*

### Key Definitions

- **Principal Component:** A linear combination of original features that captures maximum variance
- **Explained Variance Ratio:** Proportion of total variance explained by each principal component
- **Scree Plot:** Plot of eigenvalues or explained variance vs. component number
- **Loading:** Correlation between original features and principal components
- **Score:** The projected data values in the principal component space
- **Singular Value Decomposition (SVD):** Alternative method to compute PCA

### Common Mistakes to Avoid

- Forgetting to standardize data before PCA (unless features are already comparable)
- Misinterpreting principal components as original features
- Choosing number of components without examining explained variance
- Assuming PCA removes noise (it actually preserves directions of maximum variance, which may include noise)
- Confusing PCA with other dimensionality reduction techniques (LDA, t-SNE, etc.)
- Overlooking that PCA creates linear combinations (may not be interpretable)
- Ignoring reconstruction error when evaluating PCA quality
- Applying PCA to categorical data without proper encoding
